In [1]:
from pathlib import Path
import pandas as pd
import re

PROJECT_ROOT = Path("/home/harielpadillasanchez/Documentos/TT/TT2")

# Archivos del notebook 14
FINAL_DIR = PROJECT_ROOT / "FINAL_COMPARACION_36"
NB14_ALL = FINAL_DIR / "04_todos_los_sistemas_sample36.csv"

# Archivos del notebook 16
NB16_DETAIL = (
    PROJECT_ROOT
    / "outputs"
    / "evaluacion_integral_lora_vs_hiperparametros"
    / "detalle_216_cuantitativo_cualitativo.csv"
)

df14 = pd.read_csv(NB14_ALL)
df16 = pd.read_csv(NB16_DETAIL)

def norm_text(x):
    x = "" if pd.isna(x) else str(x)
    x = x.lower()
    x = re.sub(r"\s+", " ", x).strip()
    return x

df14_lora = df14[df14["system_name"].isin(["lora_llama3", "lora_mistral"])].copy()
df16_lora = df16[df16["system_name"].isin(["lora_llama3", "lora_mistral"])].copy()

# Usar row_id si existe en ambos; si no, usar sample36_id.
print("Columnas df14:", df14_lora.columns.tolist())
print("Columnas df16:", df16_lora.columns.tolist())

# En df14 existe row_id; en df16 tal vez no, pero sí sample36_id.
# Creamos sample36_id por orden dentro de cada sistema en df14.
df14_lora = df14_lora.sort_values(["system_name", "row_id"]).reset_index(drop=True)
df14_lora["sample36_id"] = df14_lora.groupby("system_name").cumcount() + 1

df16_lora = df16_lora.sort_values(["system_name", "sample36_id"]).reset_index(drop=True)

pairs = [
    ("lora_llama3", "lora_llama3"),
    ("lora_llama3", "lora_mistral"),
    ("lora_mistral", "lora_llama3"),
    ("lora_mistral", "lora_mistral"),
]

rows = []

for s14, s16 in pairs:
    a = df14_lora[df14_lora["system_name"] == s14][
        ["sample36_id", "generated_text"]
    ].rename(columns={"generated_text": "generated_14"})
    
    b = df16_lora[df16_lora["system_name"] == s16][
        ["sample36_id", "generated_text"]
    ].rename(columns={"generated_text": "generated_16"})
    
    comp = a.merge(b, on="sample36_id", how="inner")
    
    comp["same"] = (
        comp["generated_14"].apply(norm_text)
        ==
        comp["generated_16"].apply(norm_text)
    )
    
    rows.append({
        "sistema_14": s14,
        "sistema_16": s16,
        "comparaciones": len(comp),
        "textos_iguales": int(comp["same"].sum()),
        "porcentaje_igual": round(comp["same"].mean() * 100, 2),
    })

compare_cross = pd.DataFrame(rows)

display(compare_cross)

Columnas df14: ['row_id', 'prompt_family', 'owner', 'model_key', 'config_label', 'ruleset', 'source_text', 'reference_text', 'generated_text', 'sari', 'bertscore_f1', 'rougeL_f', 'compression_ratio_eval', 'exact_copy', 'system_family', 'system_name']
Columnas df16: ['sample36_id', 'texto_eval_id', 'system_name', 'system_family', 'source_text', 'reference_text', 'generated_text', 'src_words', 'pred_words', 'ref_words', 'src_sentences', 'pred_sentences', 'ref_sentences', 'sari', 'bleu', 'fernandez_huerta_pred', 'fernandez_huerta_src', 'fernandez_huerta_delta', 'compression_ratio_eval', 'sentence_splits', 'levenshtein_similarity', 'exact_copy', 'additions_proportion', 'deletions_proportion', 'inflesz_pred', 'inflesz_src', 'inflesz_delta', 'rouge1_f', 'rouge2_f', 'rougeL_f', 'bertscore_f1', 'sbert_similarity', 'score_cualitativo_1_5', 'score_cualitativo_100', 'aplico_regla_cero', 'score_fidelidad', 'score_comprension', 'score_fluidez', 'score_lexico', 'score_tono', 'score_estructura', 'sco

,sistema_14,sistema_16,comparaciones,textos_iguales,porcentaje_igual
0,lora_llama3,lora_llama3,36,8,22.22
1,lora_llama3,lora_mistral,36,36,100.00
2,lora_mistral,lora_llama3,36,36,100.00
3,lora_mistral,lora_mistral,36,8,22.22


In [2]:
from pathlib import Path
import pandas as pd
import re

PROJECT_ROOT = Path("/home/harielpadillasanchez/Documentos/TT/TT2")
FINAL_DIR = PROJECT_ROOT / "FINAL_COMPARACION_36"

LORA_LLAMA_CSV = FINAL_DIR / "02_lora_llama3_sample36.csv"
LORA_MISTRAL_CSV = FINAL_DIR / "03_lora_mistral_sample36.csv"

NB16_DETAIL = (
    PROJECT_ROOT
    / "outputs"
    / "evaluacion_integral_lora_vs_hiperparametros"
    / "detalle_216_cuantitativo_cualitativo.csv"
)

print("LORA_LLAMA_CSV:", LORA_LLAMA_CSV, "| existe:", LORA_LLAMA_CSV.exists())
print("LORA_MISTRAL_CSV:", LORA_MISTRAL_CSV, "| existe:", LORA_MISTRAL_CSV.exists())
print("NB16_DETAIL:", NB16_DETAIL, "| existe:", NB16_DETAIL.exists())

df_llama_original = pd.read_csv(LORA_LLAMA_CSV)
df_mistral_original = pd.read_csv(LORA_MISTRAL_CSV)
df16 = pd.read_csv(NB16_DETAIL)

def norm_text(x):
    x = "" if pd.isna(x) else str(x)
    x = x.lower()
    x = re.sub(r"\s+", " ", x).strip()
    return x

def prepare_original(df, expected_name):
    out = df.copy()
    out = out.sort_values("row_id").reset_index(drop=True) if "row_id" in out.columns else out.reset_index(drop=True)
    out["sample36_id"] = range(1, len(out) + 1)
    out["expected_source_file"] = expected_name
    return out[["sample36_id", "generated_text", "expected_source_file"]]

orig_llama = prepare_original(df_llama_original, "02_lora_llama3_sample36.csv")
orig_mistral = prepare_original(df_mistral_original, "03_lora_mistral_sample36.csv")

orig = pd.concat([orig_llama, orig_mistral], ignore_index=True)

df16_lora = df16[df16["system_name"].isin(["lora_llama3", "lora_mistral"])].copy()
df16_lora = df16_lora[["sample36_id", "system_name", "generated_text"]].copy()

rows = []

for source_file in orig["expected_source_file"].unique():
    for system16 in ["lora_llama3", "lora_mistral"]:
        a = orig[orig["expected_source_file"] == source_file][["sample36_id", "generated_text"]].rename(
            columns={"generated_text": "generated_original"}
        )
        b = df16_lora[df16_lora["system_name"] == system16][["sample36_id", "generated_text"]].rename(
            columns={"generated_text": "generated_16"}
        )

        comp = a.merge(b, on="sample36_id", how="inner")
        comp["same"] = comp["generated_original"].apply(norm_text) == comp["generated_16"].apply(norm_text)

        rows.append({
            "archivo_original": source_file,
            "sistema_en_notebook16": system16,
            "comparaciones": len(comp),
            "textos_iguales": int(comp["same"].sum()),
            "porcentaje_igual": round(comp["same"].mean() * 100, 2),
        })

validacion_origen = pd.DataFrame(rows)
display(validacion_origen)

LORA_LLAMA_CSV: /home/harielpadillasanchez/Documentos/TT/TT2/FINAL_COMPARACION_36/02_lora_llama3_sample36.csv | existe: True
LORA_MISTRAL_CSV: /home/harielpadillasanchez/Documentos/TT/TT2/FINAL_COMPARACION_36/03_lora_mistral_sample36.csv | existe: True
NB16_DETAIL: /home/harielpadillasanchez/Documentos/TT/TT2/outputs/evaluacion_integral_lora_vs_hiperparametros/detalle_216_cuantitativo_cualitativo.csv | existe: True


,archivo_original,sistema_en_notebook16,comparaciones,textos_iguales,porcentaje_igual
0,02_lora_llama3_sample36.csv,lora_llama3,36,8,22.22
1,02_lora_llama3_sample36.csv,lora_mistral,36,36,100.00
2,03_lora_mistral_sample36.csv,lora_llama3,36,36,100.00
3,03_lora_mistral_sample36.csv,lora_mistral,36,8,22.22


In [3]:
from pathlib import Path
import pandas as pd
import sys

PROJECT_ROOT = Path("/home/harielpadillasanchez/Documentos/TT/TT2")
FINAL_DIR = PROJECT_ROOT / "FINAL_COMPARACION_36"

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.evaluation.metrics import evaluate_dataframe

LORA_LLAMA_CSV = FINAL_DIR / "02_lora_llama3_sample36.csv"
LORA_MISTRAL_CSV = FINAL_DIR / "03_lora_mistral_sample36.csv"

df_llama = pd.read_csv(LORA_LLAMA_CSV)
df_mistral = pd.read_csv(LORA_MISTRAL_CSV)

df_llama["system_name"] = "lora_llama3"
df_mistral["system_name"] = "lora_mistral"

df_lora_original = pd.concat([df_llama, df_mistral], ignore_index=True)

# Asegurar columnas mínimas
cols_needed = ["system_name", "source_text", "reference_text", "generated_text"]
missing = [c for c in cols_needed if c not in df_lora_original.columns]

if missing:
    raise ValueError(f"Faltan columnas: {missing}")

df_lora_recalc = evaluate_dataframe(
    df_lora_original[cols_needed].copy(),
    source_col="source_text",
    pred_col="generated_text",
    ref_col="reference_text",
    compute_bertscore=True,
    compute_sbert=False,
    bertscore_lang="es",
)

metrics = [
    "sari",
    "bleu",
    "bertscore_f1",
    "rouge1_f",
    "rouge2_f",
    "rougeL_f",
    "compression_ratio_eval",
    "exact_copy",
    "levenshtein_similarity",
    "additions_proportion",
    "deletions_proportion",
]

summary_recalc_original = (
    df_lora_recalc
    .groupby("system_name", as_index=False)
    .agg(**{m: (m, "mean") for m in metrics if m in df_lora_recalc.columns})
)

summary_recalc_original["leader_score"] = (
    0.60 * summary_recalc_original["sari"] +
    0.40 * (summary_recalc_original["bertscore_f1"] * 100)
)

display(summary_recalc_original)

/home/harielpadillasanchez/Documentos/TT/TT2/.venv-bloom/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


,system_name,sari,bleu,bertscore_f1,rouge1_f,rouge2_f,rougeL_f,compression_ratio_eval,exact_copy,levenshtein_similarity,additions_proportion,deletions_proportion,leader_score
0,lora_llama3,45.340000,43.802869,0.895692,0.716483,0.578017,0.686885,0.835316,0.138889,0.811545,0.063352,0.221930,63.031661
1,lora_mistral,41.176172,41.995827,0.891969,0.709323,0.572314,0.679365,0.868639,0.194444,0.820872,0.062510,0.181661,60.384444


In [4]:
metrics_compare = [
    "sari",
    "bleu",
    "bertscore_f1",
    "rougeL_f",
    "compression_ratio_eval",
    "exact_copy",
]

summary_saved_14 = (
    df_lora_original
    .groupby("system_name", as_index=False)
    .agg(**{
        m: (m, "mean")
        for m in metrics_compare
        if m in df_lora_original.columns
    })
)

summary_saved_14["leader_score"] = (
    0.60 * summary_saved_14["sari"] +
    0.40 * (summary_saved_14["bertscore_f1"] * 100)
)

print("Métricas guardadas en CSV del notebook 14:")
display(summary_saved_14)

print("Métricas recalculadas con metrics.py:")
display(summary_recalc_original)

Métricas guardadas en CSV del notebook 14:


,system_name,sari,bertscore_f1,rougeL_f,compression_ratio_eval,exact_copy,leader_score
0,lora_llama3,44.218183,0.895692,0.686885,0.835316,0.138889,62.358570
1,lora_mistral,39.990620,0.891969,0.679365,0.868639,0.194444,59.673112


Métricas recalculadas con metrics.py:


,system_name,sari,bleu,bertscore_f1,rouge1_f,rouge2_f,rougeL_f,compression_ratio_eval,exact_copy,levenshtein_similarity,additions_proportion,deletions_proportion,leader_score
0,lora_llama3,45.340000,43.802869,0.895692,0.716483,0.578017,0.686885,0.835316,0.138889,0.811545,0.063352,0.221930,63.031661
1,lora_mistral,41.176172,41.995827,0.891969,0.709323,0.572314,0.679365,0.868639,0.194444,0.820872,0.062510,0.181661,60.384444
